In [1]:
# connect colab to drive
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pymatgen torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 5.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 7.2 MB/s eta 0:00:00
  Created w

## Get the data

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import CGConv, global_mean_pool, GATConv
from torch_geometric.data import Data, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
BASE_DIR = '/content/drive/MyDrive/Project'

train_graphs = torch.load(f"{BASE_DIR}/training.pt", weights_only=False)
val_graphs = torch.load(f"{BASE_DIR}/validation.pt", weights_only=False)
test_graphs  = torch.load(f"{BASE_DIR}/test.pt", weights_only=False)

# Create DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)

/tmp/ipykernel_899/2293329524.py:9: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
/tmp/ipykernel_899/2293329524.py:10: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False)
/tmp/ipykernel_899/2293329524.py:11: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)


# Models

### Model 1

In [12]:
# CharlesCGCNN definition
class CharlesCGCNN(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, hidden_dim,
                global_dim, num_classes):
        super().__init__()
        # Initial embedding of raw atom features
        self.atm_emb = nn.Embedding(118, hidden_dim)
        self.node_emb = nn.Linear(hidden_dim + node_feat_dim - 1, hidden_dim)

        # CGConv layers – integer channels, so in_dim = out_dim = hidden_dim
        self.conv1 = CGConv(hidden_dim, dim=edge_feat_dim)
        self.conv2 = CGConv(hidden_dim, dim=edge_feat_dim)

        # Batch normalisation for stable training
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Classifier: pooled atom embedding + global features
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u
        if u.dim() == 3:               # in case u has shape [B,1,12]
            u = u.squeeze(1)

        # Embed atoms
        z = x[:, 0].long()     # Get the atomic numbers only
        x_num = x[:, 1:]       # Get the other features

        z_emb = self.atm_emb(z)
        x = torch.cat([z_emb, x_num], dim=1)
        x = F.relu(self.node_emb(x))    # [N, hidden_dim]

        # First convolution
        x = self.conv1(x, edge_index, edge_attr)
        x = self.bn1(x)
        x = F.relu(x)

        # Second convolution
        x = self.conv2(x, edge_index, edge_attr)
        x = self.bn2(x)
        x = F.relu(x)

        # Global mean pool → graph-level vector
        x = global_mean_pool(x, batch)  # [B, hidden_dim]

        # Concatenate global features

        if u.dim() == 1:
            u = u.view(x.size(0), -1)

        out = torch.cat([x, u], dim=1)  # [B, hidden_dim+12]
        out = self.fc(out)               # [B, 2]
        return out

torch.manual_seed(42)
charles_cgcnn = CharlesCGCNN(
    node_feat_dim=8,
    edge_feat_dim=9,
    hidden_dim=64,
    global_dim=12,
    num_classes=2
).to(device)

### Model 2

In [6]:
# MEGNet block
from torch_geometric.utils import scatter

class MEGNetBlock(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim + 2*node_dim + global_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim + hidden + global_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )
        self.global_mlp = nn.Sequential(
            nn.Linear(global_dim + hidden + hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden)
        )

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u

        src, dst = edge_index
        # Edge update
        u_edge = u[batch[src]]
        edge_input = torch.cat([edge_attr, x[src], x[dst], u_edge], dim=1)
        edge_attr_new = self.edge_mlp(edge_input)

        # Node update
        agg_edges = scatter(edge_attr_new, dst,
                            dim=0, dim_size=x.size(0), reduce='mean')
        u_node = u[batch]
        node_input = torch.cat([x, agg_edges, u_node], dim=1)
        x_new = self.node_mlp(node_input)

        # Global update
        mean_nodes = scatter(x_new, batch,
                             dim=0, dim_size=u.size(0), reduce='mean')
        mean_edges = scatter(edge_attr_new, batch[src],
                             dim=0, dim_size=u.size(0), reduce='mean')
        global_input = torch.cat([u, mean_nodes, mean_edges], dim=1)
        u_new = self.global_mlp(global_input)

        return x_new, edge_attr_new, u_new

torch.manual_seed(42)
megnet_block = MEGNetBlock(node_dim=8, edge_dim=9, global_dim=12, hidden=64).to(device)


### Model 3

In [10]:
class CharlesAttentionGNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, heads=4, global_dim=12, num_classes=2):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)

        # GAT layer 1: multi-head, concatenated
        self.gat1 = GATConv(hidden_dim, hidden_dim,
                            heads=heads, edge_dim=edge_feat_dim, concat=True)
        # GAT layer 2: single head, average
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim,
                            heads=1, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data, return_attention=False):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        batch = data.batch
        u = data.u
        if u.dim() == 3:
            u = u.squeeze(1)

        x = F.relu(self.node_emb(x))          # [N, hidden_dim]

        if return_attention:
            x, (att_edge, att_weight) = self.gat1(
                x, edge_index, edge_attr, return_attention_weights=True
            )
        else:
            x = self.gat1(x, edge_index, edge_attr)
        x = F.elu(self.bn1(x))                # [N, hidden_dim*heads]

        x = self.gat2(x, edge_index)
        x = F.elu(self.bn2(x))                # [N, hidden_dim]

        x = global_mean_pool(x, batch)        # [B, hidden_dim]
        out = torch.cat([x, u], dim=1)
        out = self.fc(out)

        if return_attention:
            return out, att_edge, att_weight
        return out

torch.manual_seed(42)
charles_attn = CharlesAttentionGNN(node_feat_dim=8, edge_feat_dim=9,hidden_dim=64, heads=4, global_dim=12, num_classes=2).to(device)


### Model 4

In [ ]:
# CharlesMEGNet definition
class CharlesMEGNet(nn.Module):
    def __init__(self, node_feat_dim, edge_feat_dim, global_dim, hidden_dim, n_blocks, num_classes):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)
        self.edge_emb = nn.Linear(edge_feat_dim, hidden_dim)

        self.blocks = nn.ModuleList([
            MEGNetBlock(hidden_dim, hidden_dim,
                        global_dim if i==0 else hidden_dim,
                        hidden_dim)
            for i in range(n_blocks)
        ])

        self.pool = Set2Set(hidden_dim, processing_steps=3)
        self.fc = nn.Linear(2*hidden_dim + hidden_dim, num_classes)  # 2*hidden from Set2Set + final global

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr
        u = data.u
        batch = data.batch
        if u.dim() == 3:
            u = u.squeeze(1)

        x = F.relu(self.node_emb(x))
        edge_attr = F.relu(self.edge_emb(edge_attr))

        for block in self.blocks:
            x, edge_attr, u = block(x, edge_index, edge_attr, u, batch)

        x_pooled = self.pool(x, batch)          # [B, 2*hidden]
        out = torch.cat([x_pooled, u], dim=1)   # [B, 2*hidden + hidden]
        out = self.fc(out)
        return out

torch.manual_seed(42)
charles_megnet = CharlesMEGNet(node_feat_dim=8, edge_feat_dim=9,global_dim=12, hidden_dim=64, n_blocks=3, num_classes=2).to(device)

# Train the Model

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [7]:
def train(model, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()

        out = model(data)
        loss = loss_fn(out, data.y)

        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
    return total_loss / len(train_loader.dataset)

def evaluate(model, loader, loss_fn):
    preds, probs, labels = [], [], []

    model.eval()
    total_loss = 0
    with torch.no_grad():
        for data in loader:
            data = data.to(device)

            out = model(data)
            loss = loss_fn(out, data.y)

            total_loss += loss.item() * data.num_graphs

            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out, dim=1)[:,1].cpu().numpy())
            labels.extend(data.y.cpu().numpy())

    eval_loss = total_loss / len(loader.dataset)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    return eval_loss, acc, f1, auc




In [13]:
# Train model
# Model 1
the_model = charles_cgcnn

# Model 2
# the_model = megnet_block

# Model 3
# the_model = charles_attn

# Model 4
# the_model = charles_megnet

optimizer = torch.optim.Adam(the_model.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.CrossEntropyLoss()
# loss_fn = nn.HingeEmbeddingLoss()
# loss_fn = nn.BCEWithLogitsLoss()
epochs = 30

train_losses = []
val_losses, accs, f1s, aucs = [], [], [], []

for epoch in range(1, epochs+1):
    train_loss = train(the_model, optimizer, loss_fn)
    val_loss, acc, f1, auc = evaluate(the_model,val_loader, loss_fn)


    print(f"Epoch: {epoch:3d}/{epochs} | Train_Loss: {train_loss:.4f} | Validation_Loss: {val_loss:.4f} | Accuracy: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

    # Add the accuracy scores to a list for future evaluation
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

# Test model
test_loss, acc, f1, auc = evaluate(the_model, test_loader, loss_fn)
print(f'Test Loss: {test_loss:.4f}\n Accuracy: {acc}\n F1_ Score: {f1}\n ROC_AUC_Score: {auc}')

NameError: name 'accuracy_score' is not defined

In [ ]:
import matplotlib.pyplot as plt
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training Loss Graph
axes[0].plot(train_losses, color='steelblue')

axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# Validation metrics
axes[1].plot(accs, label='Accuracy', color='green')
axes[1].plot(f1s, label='F1', color='orange')
axes[1].plot(aucs, label='AUC', color='red')

axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics')
axes[1].legend()
axes[1].set_ylim([0.5, 1.05])
plt.tight_layout()
plt.show()

In [ ]:
# Save the best model for feature analysis